In [2]:
import pandas as pd
import altair as alt

df = pd.read_csv("student_data_cleaned.csv")

In [28]:
# Altair 1: Boxplot of Study Hours Per Day by Major

boxplot = alt.Chart(df).mark_boxplot(
    size=50,
    outliers=alt.MarkConfig(size=60),
    color="#67198e"
).encode(
    y = alt.Y("Major:N", title="Major"),
    x = alt.X("Study_Hours_Per_Day:Q", title="Study Hours Per Day"),
    # color = alt.Color("Major:N", legend=None),
    tooltip = alt.Tooltip(["Major", "Study_Hours_Per_Day:Q"])
).properties(
    title = "Student Study Hours Per Day by Major",
    width=600,
    height=400
)

# boxplot.show()

boxplot.save("boxplot_study_hours_per_day_by_major.html")

In [9]:
# Altair 2: Scatter plot of Sleep Hours vs GPA Change with linked bar chart

# Radio button for major type selection
input_radio = alt.binding_radio(
    options=[None, "Technical", "Non-technical"],
    labels=["Both", "Technical", "Non-technical"],
    name="Major Type: "
)
type_select = alt.selection_point(fields=["Major_Type"], bind=input_radio)

# Age slider for filtering by age
age_slider = alt.binding_range(min=18, max=24, step=1, name="Max Age: ")
age_param = alt.param(value=24, bind=age_slider)

# Brush for linked bar
brush = alt.selection_interval()

scatter = alt.Chart(df).mark_circle(opacity=0.5, size=60).encode(
    x=alt.X("Sleep_Hours_Per_Day:Q", title="Sleep Hours", scale=alt.Scale(domain=[4, 10])),
    y=alt.Y("GPA_Change:Q", title="GPA Change"),
    color=alt.condition(
        brush,
        alt.Color("Major_Type:N", scale=alt.Scale(
            domain=["Technical", "Non-technical"],
            range=["#0D55F0", "#D7274A"]
        )),
        alt.value("lightgrey")
    ),
    tooltip=[
        alt.Tooltip("Major:N"),
        alt.Tooltip("Sleep_Hours_Per_Day:Q", title="Sleep Hours"),
        alt.Tooltip("GPA_Change:Q", title="GPA Change", format=".2f"),
        alt.Tooltip("Age:Q")
    ]
).add_params(
    brush, age_param, type_select
).transform_filter(
    alt.datum.Age <= age_param
).transform_filter(
    type_select
).properties(
    title="Sleep Hours vs GPA Change",
    width=600,
    height=300
)

bars = alt.Chart(df).mark_bar().encode(
    x=alt.X("Major:N", title="Major"),
    y=alt.Y("count():Q", title="Count of Students"),
    color=alt.Color("Major_Type:N", scale=alt.Scale(
        domain=["Technical", "Non-technical"],
        range=["#0D55F0", "#D7274A"]), 
        legend=alt.Legend(title="Major Type"))
).transform_filter(
    brush
).transform_filter(
    alt.datum.Age <= age_param
).transform_filter(
    type_select
).properties(
    title="Students in Selected Region",
    width=500,
    height=150
)

zero_line = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(
    color="black", strokeDash=[6, 4], size=1.5
).encode(y="y:Q")

# scatter & bars

((scatter + zero_line) & bars).save("scatter_bar_linked_sleep_gpa.html")

In [ ]:
# Altair 3: Scatter plot of Study Hours Per Day vs GPA Change

scatter_study = alt.Chart(df).mark_circle(
    opacity=0.4, 
    size=40, 
    color="#67198e"
    ).encode(
    x=alt.X("Study_Hours_Per_Day:Q", title="Study Hours Per Day"),
    y=alt.Y("GPA_Change:Q", title="GPA Change"),
    tooltip=[
        alt.Tooltip("Major:N"),
        alt.Tooltip("Study_Hours_Per_Day:Q", title="Study Hours", format=".1f"),
        alt.Tooltip("GPA_Change:Q", title="GPA Change", format=".2f"),
    ]
)

trendline = scatter_study.transform_regression(
    "Study_Hours_Per_Day", "GPA_Change", as_=["Study_Hours_Per_Day", "GPA_Change"]
).mark_line(color="#F97910", size=3)

# y = 0 reference line
zero_line = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(
    color="black", strokeDash=[6, 4], size=1.5
).encode(y="y:Q")


(scatter_study + trendline + zero_line).properties(
    title="Study Hours Per Day vs GPA Change",
    width=600,
    height=400
).save("scatter_study_hours_gpa_change.html")